# 1. Cross-validation folds

Make the folds for one **experiment**: one dataset with one set of conditions. Every model in the benchmark
reads these same files, which is what makes the comparison fair.

The code is `aircheck_benchmark_core.folds` — the same as `aircheck-bench make-folds` on the command line, and
the `cluster` method is the one from `ml-pipeline-aircheck`. This replaces
[../../archive-to-be-deleted/notebooks/cluster_cv_folds.ipynb](../../archive-to-be-deleted/notebooks/cluster_cv_folds.ipynb) and `stratified_cv.py`.

**Environment:** the `biomed-multiview` kernel, with aircheck-benchmark-core installed (see [../README.md](../README.md)).

**Output** in `aircheck/output/<EXPERIMENT>/folds/`:
```
<train name>_<method>_cv_data.csv     the training molecules the fold indices refer to
<train name>_<method>_cv_folds.npz    fold0_train, fold0_test, fold1_train, ...
<train name>_<method>_cv_folds.json   method, settings, checksum of the train file, per-fold counts
```

## Settings

In [ ]:
EXPERIMENT = "sample-benchmark"                    # folder under aircheck/output/
TRAIN_PATH = "../../sample_data/train.parquet"    # relative to this notebook, or an absolute path

CV_METHOD = "cluster"        # "cluster" (chemistry held out), "stratified" or "random"
N_FOLDS = None               # None = the method's default (5)
SMILES_COL, LABEL_COL = "SMILES", "LABEL"
FINGERPRINT_COL = "ECFP4"    # cluster folds only: the fingerprint the actives are clustered on

In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Make the benchmark code importable (works from this folder or from the repo root)
BENCHMARK = next(p / rel for p in [Path.cwd(), *Path.cwd().parents] for rel in ("", "aircheck/benchmark")
                 if (p / rel / "methods" / "common.py").exists())
sys.path.insert(0, str(BENCHMARK))
AIRCHECK = BENCHMARK.parent
EXP_DIR = AIRCHECK / "output" / EXPERIMENT
print(f"Benchmark code : {BENCHMARK}")
print(f"Experiment     : {EXP_DIR}")

## Make the folds

Cluster folds compute a Tanimoto distance matrix over the actives, so this takes a little while on a large
training file. Running it again overwrites the files; use another `EXPERIMENT` for other fold settings.

In [ ]:
from aircheck_benchmark_core.folds import make_folds

paths = make_folds(TRAIN_PATH, EXP_DIR / "folds", method=CV_METHOD, smiles_col=SMILES_COL,
                   label_col=LABEL_COL, fingerprint_col=FINGERPRINT_COL,
                   **({"n_folds": N_FOLDS} if N_FOLDS is not None else {}))
for what, path in paths.items():
    print(f"{what:6s} {path}")

## What came out

In [ ]:
import json

info = json.loads(paths["info"].read_text())
print(f"{info['method']} folds, settings {info['settings']}")
print(f"{info['n_rows']:,} molecules, {info['n_pos']:,} actives, {info['n_folds']} fold(s)")
print(f"core version {info['core_version']}")
pd.DataFrame(info["folds"])

`held_out_pos` (cluster folds) counts actives from the test clusters that are in neither the fold's
training nor its test set, because the whole cluster is kept out of training.

Next: [02_finetune.ipynb](02_finetune.ipynb) or [03_embeddings_lgbm.ipynb](03_embeddings_lgbm.ipynb).